In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 定义我们的网络结构
class SimpleCNN(nn.Module):
    def __init__(self):
        """
        在这个函数里，我们定义网络中需要用到的所有“积木”（层）。
        """
        super(SimpleCNN, self).__init__()
        
        # --- 特征提取部分 ---
        # 流程: [卷积层 -> 池化层] x 2
        
        # 第一个卷积层 + 池化层
        # nn.Conv2d(输入通道数, 输出通道数/卷积核数量, 卷积核大小)
        # 输入是 3 通道 (RGB彩色图片), 我们让它输出 16 张特征图谱, 卷积核大小为 3x3
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1) 
        # nn.MaxPool2d(池化窗口大小, 步幅)
        # 用 2x2 的窗口进行最大池化, 步幅为 2, 这会让特征图谱的尺寸减半
        self.pool = nn.MaxPool2d(2, 2)
        
        # 第二个卷积层 + 池化层
        # 这一层的输入是上一层输出的 16 张特征图谱
        # 我们让它输出 32 张更复杂的特征图谱
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # 池化层可以共用同一个定义
        
        # --- 分类决策部分 ---
        # 流程: [全连接层] x 2
        
        # 第一个全连接层
        # nn.Linear(输入特征数, 输出特征数)
        # 输入特征数需要计算: 经过两次池化, 32x32的图片变成了 8x8。
        # 并且有 32 张特征图谱, 所以总特征数是 32 * 8 * 8 = 2048
        self.fc1 = nn.Linear(32 * 8 * 8, 120)
        
        # 第二个全连接层 (最终输出层)
        # CIFAR-10 数据集有 10 个分类, 所以最终输出 10 个数, 代表每个分类的得分
        self.fc2 = nn.Linear(120, 10)

    def forward(self, x):
        """
        在这个函数里，我们规定数据（x）是如何在这些“积木”中流动的。
        """
        # --- 特征提取的流动路线 ---
        # 输入 x -> Conv1 -> ReLU激活 -> Pool
        x = self.pool(F.relu(self.conv1(x)))
        # -> Conv2 -> ReLU激活 -> Pool
        x = self.pool(F.relu(self.conv2(x)))
        
        # --- 分类决策的流动路线 ---
        # 在进入全连接层之前，需要将二维的特征图谱“压平”成一维长条
        # x.size(0) 是批次大小, -1 表示自动计算剩余维度
        x = torch.flatten(x, 1) # 或者 x.view(-1, 32 * 8 * 8)
        
        # -> FC1 -> ReLU激活
        x = F.relu(self.fc1(x))
        # -> FC2 (最终输出)
        x = self.fc2(x)
        return x

# --- 测试我们的网络 ---
if __name__ == '__main__':
    # 创建一个网络实例
    net = SimpleCNN()
    print("网络结构:")
    print(net)
    
    # 创建一个假的输入图片数据来测试网络是否能跑通
    # PyTorch 需要的格式: (批次大小, 通道数, 高度, 宽度)
    dummy_input = torch.randn(1, 3, 32, 32)
    
    # 将假数据输入网络
    output = net(dummy_input)
    
    # 打印输出结果的形状
    print("\n输入尺寸:", dummy_input.shape)
    print("输出尺寸:", output.shape)
    print("输出结果示例 (代表10个分类的得分):", output)

网络结构:
SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=2048, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=10, bias=True)
)

输入尺寸: torch.Size([1, 3, 32, 32])
输出尺寸: torch.Size([1, 10])
输出结果示例 (代表10个分类的得分): tensor([[-0.0334, -0.0702, -0.0184,  0.1456, -0.0889,  0.0809, -0.0035,  0.0809,
          0.1068, -0.0547]], grad_fn=<AddmmBackward0>)
